In [ ]:
import os

live_root = "/kaggle/input/datasets/username/numerai-live-v52/live.parquet"
weights_cb = "/kaggle/input/datasets/username/numerai-weights-v52-dual-c0102/catboost_model_0058.cbm"
weights_mlp = "/kaggle/input/datasets/username/numerai-weights-v52-dual-c0102/mlp_model_0102.pth"
feat_path = "/kaggle/input/datasets/username/numerai-custom-features/custom_features.json"

print(f"{live_root}, {weights_cb}, {weights_mlp}, {feat_path}")

/kaggle/input/datasets/shuangsong/numerai-live-v52/live.parquet, /kaggle/input/datasets/shuangsong/numerai-weights-v52-dual-c0102/catboost_model_0058.cbm, /kaggle/input/datasets/shuangsong/numerai-weights-v52-dual-c0102/mlp_model_0102.pth, /kaggle/input/datasets/shuangsong/numerai-custom-features/custom_features.json


In [5]:
import torch
import torch.nn as nn
import pandas as pd
import json
from catboost import CatBoostRegressor

cb_model = CatBoostRegressor()
cb_model.load_model(weights_cb)

class NumeraiMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(0.1), 
            nn.Linear(in_dim, 256),
            nn.SiLU(),     
            nn.Linear(256, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze()

In [8]:
with open(feat_path, 'r') as f:
    features = json.load(f)['feature_sets']['custom_features']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp = NumeraiMLP(len(features) + 1).to(device)
mlp.load_state_dict(torch.load(weights_mlp, map_location=device))
mlp.eval()

live_df = pd.read_parquet(live_root, columns=features).fillna(0.5)
X_np = live_df[features].values.astype("float32").copy()
cb_preds = cb_model.predict(X_np)
cb_preds_t = torch.tensor(cb_preds, device=device, dtype=torch.float32).unsqueeze(1)
X_mlp = torch.cat([torch.from_numpy(X_np).to(device), cb_preds_t], dim=1)

with torch.no_grad():
    preds = mlp(X_mlp).cpu().numpy().flatten()

pd.DataFrame({"id": live_df.index, "prediction": pd.Series(preds).rank(pct=True).values}).to_csv("submission_dual.csv", index=False)